# Notebook 2 — LangChain Basics

In Notebook 1 we sent prompts straight to the model with `ask()`. Real apps need more than
that: the same prompt with many inputs, several model calls in a row, and answers turned into
data. **LangChain** is a Python library that gives you small building blocks for these jobs,
and a simple way to connect them.

**What you will learn**

1. Chat models and message objects.
2. Three ways to run anything: `invoke`, `batch`, and `stream`.
3. Prompt templates.
4. Chains with the `|` operator.
5. Output parsers and structured output.
6. Sequential and parallel chains.

**Before you start**

```bash
ollama pull gpt-oss:20b
uv sync
```

Then choose the `.venv` Python kernel for this notebook. ⏱️ About 45 minutes.

## 1. Setup

`get_chat_model()` from the `roadmap` package returns a LangChain chat model for the local
Ollama server. It uses `gpt-oss:20b` with short thinking (`reasoning="low"`) by default.

In [1]:
import time
from pprint import pprint

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import (
    CommaSeparatedListOutputParser,
    JsonOutputParser,
    StrOutputParser,
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from pydantic import BaseModel, Field

from roadmap import get_chat_model

llm = get_chat_model()
print(type(llm).__name__, "->", llm.model)

ChatOllama -> gpt-oss:20b


## 2. Chat models and messages

LangChain wraps every model provider (Ollama, OpenAI, Anthropic, ...) in a **chat model**
class. They all have the same methods, so you can switch providers without rewriting your
app. We use `ChatOllama`.

Instead of plain dictionaries, LangChain uses **message objects**:

| Class | Role | Meaning |
|---|---|---|
| `SystemMessage` | system | Rules for the whole conversation. |
| `HumanMessage` | user | What the person says. |
| `AIMessage` | assistant | What the model answered. |

In [2]:
response = llm.invoke(
    [
        SystemMessage(content="You suggest fitness activities in one short sentence."),
        HumanMessage(content="I like high-intensity workouts. What should I do?"),
    ]
)

print("Type:     ", type(response).__name__)
print("Answer:   ", response.content)
print("Thinking: ", response.additional_kwargs.get("reasoning_content"))
print("Tokens:   ", response.usage_metadata)

Type:      AIMessage
Answer:    Try a 30‑minute HIIT circuit: 45 s of burpees, 15 s rest, 45 s of jump squats, 15 s rest, 45 s of mountain climbers, 15 s rest, repeat.
Thinking:  Need short sentence.
Tokens:    {'input_tokens': 96, 'output_tokens': 70, 'total_tokens': 166}


The answer comes back as an `AIMessage`. The text is in `.content`, the model's thinking is
kept apart in `additional_kwargs`, and `.usage_metadata` counts the tokens.

**The model has no memory.** Each call starts from zero. To continue a conversation, you send
the earlier messages again, including the model's own answer:

In [3]:
conversation = [
    SystemMessage(content="You suggest fitness activities in one short sentence."),
    HumanMessage(content="I like high-intensity workouts. What should I do?"),
    AIMessage(content="Try a CrossFit class."),
    HumanMessage(content="How often should I go?"),
]

print(llm.invoke(conversation).content)

Aim for 3–4 sessions per week.


"How often should I go?" only makes sense because the history is included. Notebook 3 shows
how to manage this history for you (memory).

For a single question you can also pass a plain string. LangChain turns it into a
`HumanMessage`:

In [4]:
print(llm.invoke("What month follows June? Reply with one word.").content)

July


## 3. Three ways to run: invoke, batch, stream

![Three ways to run any model, prompt, or chain](assets/runnable_interface.svg)

In LangChain, models, prompts, parsers, and chains are all **runnables**. Every runnable has
the same three methods:

| Method | Use it when |
|---|---|
| `invoke(input)` | You have one input and want the full answer. |
| `batch([inputs])` | You have many inputs. They are sent at the same time. |
| `stream(input)` | You want to show the answer piece by piece while it is written. |

`batch` takes a list and returns the answers in the same order:

In [5]:
questions = [
    f"What is the capital of {country}? Reply with one word."
    for country in ["Italy", "Peru", "Kenya", "Norway"]
]

start = time.perf_counter()
answers = [message.content for message in llm.batch(questions)]
print(answers, f"{time.perf_counter() - start:.1f}s")

['Rome', 'Lima', 'Nairobi', 'Oslo'] 1.9s


> **Is `batch` faster?** It depends on the server. LangChain sends all requests at the same
> time. Hosted APIs answer them in parallel, so `batch` is much faster there. A local Ollama
> server answers one request per model at a time by default, so the time is about the same
> as a loop. You can raise `OLLAMA_NUM_PARALLEL`, but on a laptop the requests still share
> one GPU: in our test on an M1 Max, 2 parallel slots made `batch` only about 1.2× faster.
> Either way, your code stays short.

Now `stream`. A reasoning model first thinks, and during that time the text pieces are empty,
so we only print pieces that have text:

In [6]:
pieces = []
for chunk in llm.stream("Name three planets, separated by commas."):
    if chunk.content:
        pieces.append(chunk.content)
        print(chunk.content, end="|", flush=True)

print(f"\n\n{len(pieces)} pieces with text")

Mars|

,|

 Jupiter|

,|

 Neptune|



5 pieces with text


The `|` marks show where each piece ends. Chat apps print these pieces as they arrive, so the
user sees the answer grow instead of waiting.

## 4. Prompt templates

A **prompt template** is a prompt with named blanks such as `{topic}`. You saw the idea in
Notebook 1. LangChain's templates are runnables too, so they can be part of a chain.

`PromptTemplate` builds one text prompt:

In [7]:
simple_prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}.")

print("Blanks:", simple_prompt.input_variables)
print("Filled:", simple_prompt.invoke({"adjective": "funny", "topic": "cats"}).to_string())

Blanks: ['adjective', 'topic']
Filled: Tell me one funny joke about cats.


`ChatPromptTemplate` builds a list of messages. This is what you normally use with chat
models:

In [8]:
chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant who knows a lot about {topic}."),
        ("user", "Tell me one short fact about {topic}."),
    ]
)

for message in chat_prompt.invoke({"topic": "octopuses"}).to_messages():
    print(f"{type(message).__name__:<14} {message.content}")

SystemMessage  You are a helpful assistant who knows a lot about octopuses.
HumanMessage   Tell me one short fact about octopuses.


**Partial variables** fill some blanks in advance. This is handy for text that never changes,
such as format rules:

In [9]:
reply_prompt = PromptTemplate.from_template("Answer in {language}. Question: {question}").partial(
    language="French"
)

print(reply_prompt.invoke({"question": "What time is it?"}).to_string())

Answer in French. Question: What time is it?


`MessagesPlaceholder` leaves room for a whole **list** of messages, for example the
conversation history. We will use it for memory in Notebook 3.

In [10]:
history_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder("history"),
        ("user", "{question}"),
    ]
)

filled = history_prompt.invoke(
    {
        "history": [HumanMessage(content="My name is Ada."), AIMessage(content="Hi Ada!")],
        "question": "What is my name?",
    }
)
for message in filled.to_messages():
    print(f"{type(message).__name__:<14} {message.content}")

SystemMessage  You are a helpful assistant.
HumanMessage   My name is Ada.
AIMessage      Hi Ada!
HumanMessage   What is my name?


## 5. Chains with the `|` operator

![A chain is a pipeline of small steps](assets/lcel_chain.svg)

> **Chain:** a pipeline of steps where the output of one step is the input of the next.
>
> **LCEL (LangChain Expression Language):** the way you build chains in LangChain, by
> connecting runnables with the `|` sign.

The most common chain has three steps: prompt template → chat model → output parser.
`StrOutputParser` simply takes the text out of the `AIMessage`.

In [11]:
joke_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a comedian. Keep every joke to two short lines."),
        ("user", "Tell me one {adjective} joke about {topic}."),
    ]
)
joke_chain = joke_prompt | llm | StrOutputParser()

print(joke_chain.invoke({"adjective": "funny", "topic": "cats"}))

Why did the cat bring a ladder?  
Because it heard the purr‑formance was on a higher level!


To see what really happens, run the three steps one by one and look at the type of each
result:

In [12]:
step_1 = joke_prompt.invoke({"adjective": "funny", "topic": "dogs"})
step_2 = llm.invoke(step_1)
step_3 = StrOutputParser().invoke(step_2)

print("1. prompt template ->", type(step_1).__name__, step_1.to_messages())
print("2. chat model      ->", type(step_2).__name__)
print("3. output parser   -> plain text:", isinstance(step_3, str))
print()
print(step_3)

1. prompt template -> ChatPromptValue [SystemMessage(content='You are a comedian. Keep every joke to two short lines.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me one funny joke about dogs.', additional_kwargs={}, response_metadata={})]
2. chat model      -> AIMessage
3. output parser   -> plain text: True

Why did the dog sit in the shade?  
Because he didn’t want to be a hot‑dog!


LangChain can also draw the chain for you:

In [13]:
joke_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOllama |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


A chain is a runnable, so `batch` and `stream` work on it too:

In [14]:
topics = [{"adjective": "short", "topic": topic} for topic in ["coffee", "Mondays", "Wi-Fi"]]

for joke in joke_chain.batch(topics):
    print("-", joke)

- Why did the coffee file a police report?  
It got mugged every morning!
- Monday’s the day we all pretend  
to be awake, but the coffee’s still a‑treat.
- Why did the Wi‑Fi break up?  
It said it needed more *signal* in its life.


## 6. Output parsers and structured output

The model always returns text. **Output parsers** turn that text into Python data.

| Tool | Returns | How it works |
|---|---|---|
| `StrOutputParser` | `str` | Takes the text out of the message. |
| `CommaSeparatedListOutputParser` | `list[str]` | Adds format rules to the prompt, then splits on commas. |
| `JsonOutputParser` | `dict` | Adds JSON rules to the prompt, then reads the JSON. |
| `llm.with_structured_output(Schema)` | a Pydantic object | The model must follow the schema while it writes. |

The first parsers rely on **format instructions**: extra text in the prompt that tells the
model how to answer. Here is what the list parser adds:

In [15]:
list_parser = CommaSeparatedListOutputParser()
print(list_parser.get_format_instructions())

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [16]:
list_prompt = PromptTemplate.from_template(
    "List five {subject}.\n{format_instructions}",
    partial_variables={"format_instructions": list_parser.get_format_instructions()},
)
list_chain = list_prompt | llm | list_parser

flavors = list_chain.invoke({"subject": "ice cream flavors"})
print(type(flavors).__name__, flavors)

list ['vanilla', 'chocolate', 'strawberry', 'mint chocolate chip', 'cookie dough']


For more fields, describe the data with a **Pydantic** class. `JsonOutputParser` turns the
class into JSON rules for the prompt:

In [17]:
class Movie(BaseModel):
    title: str = Field(description="the movie title")
    director: str = Field(description="the director's full name")
    year: int = Field(description="the year the movie came out")
    genres: list[str] = Field(description="one to three genres")


json_parser = JsonOutputParser(pydantic_object=Movie)
json_prompt = PromptTemplate.from_template(
    "Give information about the movie {movie}.\n{format_instructions}",
    partial_variables={"format_instructions": json_parser.get_format_instructions()},
)
json_chain = json_prompt | llm | json_parser

movie = json_chain.invoke({"movie": "Inception"})
print(type(movie).__name__, movie)

dict {'title': 'Inception', 'director': 'Christopher Nolan', 'year': 2010, 'genres': ['Action', 'Sci-Fi', 'Thriller']}


This works, but it depends on the model following the instructions. If it adds an extra
sentence, parsing can fail.

**`with_structured_output` is the safer way.** The schema is sent to the model, and Ollama
makes the model write output that fits it (the Message Analyzer project uses the same idea).
You get a ready Pydantic object back:

In [18]:
movie_model = llm.with_structured_output(Movie)

matrix = movie_model.invoke("Give information about the movie The Matrix (1999).")
print(type(matrix).__name__)
print(matrix.title, "|", matrix.director, "|", matrix.year, "|", ", ".join(matrix.genres))

Movie
The Matrix | The Wachowskis | 1999 | Action, Science Fiction, Cyberpunk, Philosophical Thriller


**Rule of thumb:** use `with_structured_output` when the model supports it. Use the parsers
when it does not, or when you only need plain text or a simple list.

## 7. Your own Python steps

Any Python function can be a step in a chain. Wrap it in `RunnableLambda` (inside a chain, a
plain function is wrapped for you).

In [19]:
def add_word_count(text: str) -> dict:
    return {"summary": text, "words": len(text.split())}


summary_chain = (
    PromptTemplate.from_template("Summarize in one sentence: {text}")
    | llm
    | StrOutputParser()
    | RunnableLambda(add_word_count)
)

article = (
    "Online learning platforms make education easier to reach. People can study from home, "
    "at their own speed, and often for free. Video lessons, quizzes, and forums replace parts "
    "of the classroom, although many learners still miss face-to-face contact."
)
pprint(summary_chain.invoke({"text": article}))

{'summary': 'Online learning platforms democratize education by enabling '
            'self‑paced, often free study from home with video lessons, '
            'quizzes, and forums, though many learners still miss the personal '
            'interaction of traditional classrooms.',
 'words': 30}


## 8. Sequential chains

![Sequential chain: each step adds one new key](assets/sequential_chain.svg)

Often one model call needs the result of an earlier one. `RunnablePassthrough.assign()` runs a
step and **adds** its result to the data, while keeping everything that was already there.
So later steps can use all earlier results.

In [20]:
dish_chain = (
    PromptTemplate.from_template(
        "Name one classic dish from {location}. Reply with the dish name only."
    )
    | llm
    | StrOutputParser()
)
recipe_chain = (
    PromptTemplate.from_template(
        "Write a very short home recipe for {meal}: one line of ingredients, then at "
        "most five numbered steps of one short sentence each."
    )
    | llm
    | StrOutputParser()
)
time_chain = (
    PromptTemplate.from_template(
        "How long does this recipe take in total? Reply in one short sentence.\n\n{recipe}"
    )
    | llm
    | StrOutputParser()
)

meal_planner = (
    RunnablePassthrough.assign(meal=dish_chain)
    | RunnablePassthrough.assign(recipe=recipe_chain)
    | RunnablePassthrough.assign(time=time_chain)
)

plan = meal_planner.invoke({"location": "Turkey"})
for key, value in plan.items():
    print(f"--- {key}\n{value}\n")

--- location
Turkey

--- meal
Kebap

--- recipe
**Ingredients**  
Ground lamb or beef, 1 tsp salt, 1 tsp pepper, 1 tsp paprika, 1 tsp cumin, 1 minced garlic clove, 1 tbsp olive oil, skewers.

**Steps**  
1. Mix meat with all spices, garlic, and oil until well combined.  
2. Shape the mixture onto skewers, pressing firmly.  
3. Preheat a grill or skillet over medium-high heat.  
4. Cook the kebabs, turning every 2–3 minutes, until browned and cooked through.  
5. Serve hot with flatbread, yogurt, and fresh herbs.

--- time
About 30 minutes in total.



> Older tutorials use classes like `LLMChain` and `SequentialChain`. They are the old way of
> doing the same thing. Today, chains are built with `|` and `RunnablePassthrough.assign`.

## 9. Parallel chains

![Parallel chain: same input, branches run at the same time](assets/parallel_chain.svg)

When two steps do **not** depend on each other, run them side by side with
`RunnableParallel`. Each branch gets the same input, and the result is one dictionary with a
key for each branch. (Like `batch`, the speed gain depends on the server.)

In [21]:
pros_chain = (
    PromptTemplate.from_template("Give two pros of {topic} in one line.") | llm | StrOutputParser()
)
cons_chain = (
    PromptTemplate.from_template("Give two cons of {topic} in one line.") | llm | StrOutputParser()
)
both_sides = RunnableParallel(pros=pros_chain, cons=cons_chain)

result = both_sides.invoke({"topic": "working from home"})
pprint(result, width=100)

{'cons': 'Limited social interaction and blurred boundaries between work and personal life.',
 'pros': 'Flexibility to balance personal life and work, and reduced commuting time and costs.'}


## 10. Putting it together: a review reply pipeline

A small but real workflow for an online shop:

1. **Analyse** the review into structured data (sentiment and key points).
2. **Write a reply** that uses that analysis.

Step 1 uses `with_structured_output`, step 2 a normal prompt chain, and
`RunnablePassthrough.assign` connects them.

In [22]:
class ReviewAnalysis(BaseModel):
    sentiment: str = Field(description="positive, negative, or neutral")
    key_points: list[str] = Field(description="two to four short points from the review")


analyze_prompt = PromptTemplate.from_template("Analyse this product review:\n\n{review}")
reply_prompt = PromptTemplate.from_template(
    "Write a short, friendly reply (at most three sentences) from the shop to this customer.\n"
    "If the review is negative, say sorry and offer a next step.\n\n"
    "Review: {review}\nSentiment: {sentiment}\nKey points: {key_points}"
)

review_pipeline = (
    RunnablePassthrough.assign(analysis=analyze_prompt | llm.with_structured_output(ReviewAnalysis))
    | RunnablePassthrough.assign(
        sentiment=lambda x: x["analysis"].sentiment,
        key_points=lambda x: "; ".join(x["analysis"].key_points),
    )
    | RunnablePassthrough.assign(reply=reply_prompt | llm | StrOutputParser())
)

reviews = [
    "I love this coffee maker! It brews fast, the coffee tastes great, and the timer means "
    "fresh coffee every morning.",
    "This laptop overheats after 30 minutes, the battery lasts 3 hours instead of 8, and two "
    "keys already stick. Very disappointed.",
]

for output in review_pipeline.batch([{"review": review} for review in reviews]):
    print("Sentiment: ", output["sentiment"])
    print("Key points:", output["key_points"])
    print("Reply:     ", output["reply"], "\n")

Sentiment:  positive
Key points: love this coffee maker; brews fast; coffee tastes great; timer ensures fresh coffee every morning
Reply:      Thank you so much for the glowing review! We’re thrilled to hear that your coffee maker is brewing fast, delivering great taste, and keeping your mornings fresh. Enjoy every cup, and let us know if there’s anything else we can do to make your experience even better! 

Sentiment:  negative
Key points: Laptop overheats after 30 minutes; Battery life only 3 hours vs expected 8; Two keys already sticking; Overall disappointment
Reply:      We’re really sorry to hear about the overheating, battery, and key issues—this isn’t the experience we aim for. Please let us know your order number and we’ll arrange a quick repair or replacement for you. Thank you for bringing this to our attention; we’ll do everything we can to make it right. 



## 11. Summary

| Building block | What it does | Example |
|---|---|---|
| Chat model | Talks to the LLM | `get_chat_model()` → `ChatOllama` |
| Messages | System, user, and assistant turns | `SystemMessage`, `HumanMessage`, `AIMessage` |
| Runnable methods | Run anything | `invoke`, `batch`, `stream` |
| Prompt template | Prompt with blanks | `PromptTemplate`, `ChatPromptTemplate` |
| Chain (LCEL) | Connect steps | `prompt \| llm \| parser` |
| Output parser | Text → Python data | `StrOutputParser`, `JsonOutputParser` |
| Structured output | Output that must fit a schema | `llm.with_structured_output(Schema)` |
| Custom step | Your own Python code | `RunnableLambda(function)` |
| Sequential chain | Each step adds a result | `RunnablePassthrough.assign(...)` |
| Parallel chain | Independent steps at once | `RunnableParallel(a=..., b=...)` |

➡️ **Next:** Notebook 3 — Memory, tools, and agents.